# Data Extraction and Storage

## 1. Steps to collect airport data using API and convert into SQL

In [2]:
import time
import pandas as pd

In [87]:
#Connection to the url with key
import requests
API_HOST = "aerodatabox.p.rapidapi.com"
API_KEY = "ae6852a1abmshff7e7a387a1f740p1da32ejsn8af8b9de9fbd"

headers = {
	"x-rapidapi-key": API_KEY,
	"x-rapidapi-host": API_HOST
} 

In [ ]:
#Defining a function with icao code
def fetch_airport_by_icao(icao_code):
    url=f"https://aerodatabox.p.rapidapi.com/airports/icao/{icao_code}"
    airport_response=requests.get(url,headers=headers)
    time.sleep(1)
    return airport_response.json()
    

In [ ]:
my_airports=["EGLL", "KJFK", "OMDB", "VIDP", "VOBL", "EGCC", "EDDM", "CYVR", "RJOO", "WMKP", "VTCC", "EHAM", "LFPG", "WSSS", "YSSY"]

airport_all={}

for icao in my_airports:
    try:
        airport_data=fetch_airport_by_icao(icao)
        airport_all[icao]=airport_data
        print(airport_data)
    except Exception as e:
        print(f"Error loading airport {icao}: {e}")

{'icao': 'EGLL', 'iata': 'LHR', 'shortName': 'Heathrow', 'fullName': 'London Heathrow', 'municipalityName': 'London', 'location': {'lat': 51.4706, 'lon': -0.461941}, 'elevation': {'meter': 25.3, 'km': 0.03, 'mile': 0.02, 'nm': 0.01, 'feet': 83.0}, 'country': {'code': 'GB', 'name': 'United Kingdom'}, 'continent': {'code': 'EU', 'name': 'Europe'}, 'timeZone': 'Europe/London', 'urls': {'webSite': 'http://www.heathrow.com/', 'wikipedia': 'https://en.wikipedia.org/wiki/London_Heathrow_Airport', 'twitter': 'https://x.com/HeathrowAirport', 'flightRadar': 'https://www.flightradar24.com/51.47,-0.46/14', 'googleMaps': 'https://www.google.com/maps/@51.470600,-0.461941,14z'}}
{'icao': 'KJFK', 'iata': 'JFK', 'shortName': 'John F Kennedy', 'fullName': 'New York John F Kennedy', 'municipalityName': 'New York', 'location': {'lat': 40.6398, 'lon': -73.7789}, 'elevation': {'meter': 3.96, 'km': 0.0, 'mile': 0.0, 'nm': 0.0, 'feet': 13.0}, 'country': {'code': 'US', 'name': 'United States'}, 'continent': {'

In [10]:


def extract_airport_info(airport_data):
    extracted_info = {
        'icao_code': airport_data.get('icao'),
        'iata_code': airport_data.get('iata'),
        'name': airport_data.get('fullName'),
        'city': airport_data.get('municipalityName'),
        'country': airport_data.get('country', {}).get('name'),
        'continent': airport_data.get('continent', {}).get('name'),
        'latitude': airport_data.get('location', {}).get('lat'),
        'longitude': airport_data.get('location', {}).get('lon'),
        'timezone': airport_data.get('timeZone')
    }
    return extracted_info

In [ ]:
extracted_airports_data = []
for icao, data in airport_all.items():
    extracted_airports_data.append(extract_airport_info(data))

# Create a pandas DataFrame from the extracted data
airport_df = pd.DataFrame(extracted_airports_data)
display(airport_df)

NameError: name 'airport_all' is not defined

In [3]:
#To make a connection to mysql
import mysql.connector 

connection = mysql.connector.connect(
    host = "Localhost",
    user = "root",
    password = "12345",
    )
cursor =  connection.cursor()
cursor

In [ ]:
query="create database if not exists AeroDataBox"
cursor.execute(query)
connection.commit()

In [4]:
query = "use AeroDataBox"
cursor.execute(query)

In [ ]:
query = """create table if not exists Airport(airport_id int primary key auto_increment, icao_code varchar(60) UNIQUE, iata_code varchar(60) UNIQUE, name varchar(60), city varchar(60), country varchar(60), continent varchar(60), latitude float, longitude float, timezone varchar(60))"""
cursor.execute(query)
connection.commit()

In [ ]:
columns = ",".join(airport_df.columns)
placeholders = ",".join(["%s"]*len(airport_df.columns))
query = f"insert ignore into Airport({columns}) values({placeholders})"
tuple_data = []
for i in airport_df.index:
    row = list(airport_df.loc[i].values)
    row = tuple(row)
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

## 2. Steps to collect flight data using API and convert into SQL

In [88]:
#Defining a function with icao code
def fetch_flight_by_icao(icao_code):
    url=f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao_code}/2026-05-17T20:00/2026-05-18T08:00"
    querystring = {"withLeg":"true","direction":"Both","withCancelled":"true","withCodeshared":"true","withCargo":"true","withPrivate":"true","withLocation":"false"}

    flight_response=requests.get(url,headers=headers, params=querystring)
    time.sleep(1)
    return flight_response.json()

In [92]:
def extract_flight_info(flight_data, flight_type, airport_icao):
    flight_id = flight_data.get('number', '') + '_' + flight_data.get(flight_type, {}).get('scheduledTime', {}).get('utc', '')

    # Determine origin and destination based on flight type, ensuring ICAO codes
    origin_code = ''
    destination_code = ''
    if flight_type == 'departure':
        origin_code = airport_icao # The airport from which the flights were fetched
        destination_code = flight_data.get('arrival', {}).get('airport', {}).get('icao')
    elif flight_type == 'arrival':
        origin_code = flight_data.get('departure', {}).get('airport', {}).get('icao')
        destination_code = airport_icao # The airport to which the flights were fetched

    return {
        'flight_id': flight_id,
        'flight_number': flight_data.get('number'),
        'aircraft_registration': flight_data.get('aircraft', {}).get('reg'),
        'origin_icao': origin_code,
        'destination_icao': destination_code,
        'scheduled_departure': flight_data.get('departure', {}).get('scheduledTime', {}).get('utc', ''),
        'actual_departure': flight_data.get('departure', {}).get('revisedTime', {}).get('utc', flight_data.get('departure', {}).get('scheduledTime', {}).get('utc')),#fallback extraction
        'scheduled_arrival': flight_data.get('arrival', {}).get('scheduledTime', {}).get('utc'),
        'actual_arrival': flight_data.get('arrival', {}).get('revisedTime', {}).get('utc', flight_data.get('arrival', {}).get('scheduledTime', {}).get('utc')),
        'status': flight_data.get('status'),
        'airline_code': flight_data.get('airline', {}).get('iata')
    }



In [ ]:
all_flights_data = []
list_icao = airport_df['icao_code']
flights_all={}
for icao in list_icao:
    try:
        airflight_data=fetch_flight_by_icao(icao)
        flight_all[icao] = airflight_data
        print(flight_all[icao])
    except Exception as e:
        print(f"Error loading flight {icao}: {e}")        
    

Error loading flight EGLL: name 'flight_all' is not defined
Error loading flight KJFK: name 'flight_all' is not defined
Error loading flight OMDB: name 'flight_all' is not defined
Error loading flight VIDP: name 'flight_all' is not defined
Error loading flight VOBL: name 'flight_all' is not defined
Error loading flight EGCC: name 'flight_all' is not defined
Error loading flight EDDM: name 'flight_all' is not defined
Error loading flight CYVR: name 'flight_all' is not defined
Error loading flight RJOO: name 'flight_all' is not defined
Error loading flight WMKP: name 'flight_all' is not defined
Error loading flight VTCC: name 'flight_all' is not defined
Error loading flight EHAM: name 'flight_all' is not defined


In [ ]:
for icao in list_icao:
    # Get current airport data
        curr_airport_data = flight_all[icao]
    # Extract departures
        if 'departures' in curr_airport_data:
            for flight in curr_airport_data.get('departures',[])[:20]:
                all_flights_data.append(extract_flight_info(flight, 'departure', icao)) 

# Extract arrivals
        if 'arrivals' in curr_airport_data:
            for flight in curr_airport_data.get('arrivals',[])[:20]:
                all_flights_data.append(extract_flight_info(flight, 'arrival', icao))
    
# Create DataFrame
flights_df = pd.DataFrame(all_flights_data)
display(flights_df)


,flight_id,flight_number,aircraft_registration,origin_icao,destination_icao,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,status,airline_code
0,AY 5945_2026-05-17 18:00Z,AY 5945,G-XLED,EGLL,FAOR,2026-05-17 18:00Z,2026-05-17 19:00Z,2026-05-18 05:00Z,2026-05-18 05:04Z,Departed,AY
1,BA 55_2026-05-17 18:00Z,BA 55,G-XLED,EGLL,FAOR,2026-05-17 18:00Z,2026-05-17 19:00Z,2026-05-18 05:00Z,2026-05-18 05:04Z,Departed,BA
2,AA 7106_2026-05-17 18:00Z,AA 7106,G-XLED,EGLL,FAOR,2026-05-17 18:00Z,2026-05-17 19:00Z,2026-05-18 05:00Z,2026-05-18 05:04Z,Departed,AA
3,IB 3511_2026-05-17 18:00Z,IB 3511,G-XLED,EGLL,FAOR,2026-05-17 18:00Z,2026-05-17 19:00Z,2026-05-18 05:00Z,2026-05-18 05:04Z,Departed,IB
4,KE 908_2026-05-17 18:35Z,KE 908,HL7203,EGLL,RKSI,2026-05-17 18:35Z,2026-05-17 19:02Z,2026-05-18 07:15Z,2026-05-18 06:58Z,Departed,KE
...,...,...,...,...,...,...,...,...,...,...,...
595,CI 8390_2026-05-17 10:00Z,CI 8390,NaN,NaN,YSSY,,NaN,2026-05-17 10:00Z,2026-05-17 10:14Z,Arrived,CI
596,VA 986_2026-05-17 10:40Z,VA 986,VH-IJU,YBBN,YSSY,2026-05-17 09:05Z,2026-05-17 09:00Z,2026-05-17 10:40Z,2026-05-17 10:18Z,Arrived,VA
597,QF 2183_2026-05-17 10:50Z,QF 2183,VH-QOD,YMOR,YSSY,2026-05-17 09:20Z,2026-05-17 09:20Z,2026-05-17 10:50Z,2026-05-17 10:20Z,Arrived,QF
598,NZ 7451_2026-05-17 10:50Z,NZ 7451,NaN,NaN,YSSY,,NaN,2026-05-17 10:50Z,2026-05-17 10:20Z,Arrived,NZ


In [ ]:
flights_df=flights_df.fillna("None")
flights_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   flight_id              600 non-null    str  
 1   flight_number          600 non-null    str  
 2   aircraft_registration  600 non-null    str  
 3   origin_icao            600 non-null    str  
 4   destination_icao       600 non-null    str  
 5   scheduled_departure    600 non-null    str  
 6   actual_departure       600 non-null    str  
 7   scheduled_arrival      600 non-null    str  
 8   actual_arrival         600 non-null    str  
 9   status                 600 non-null    str  
 10  airline_code           600 non-null    str  
dtypes: str(11)
memory usage: 51.7 KB


In [ ]:
query="""create table if not exists flight(flight_id varchar(60) PRIMARY KEY, flight_number varchar(60), aircraft_registration varchar(60), origin_icao varchar(60), destination_icao varchar(60), scheduled_departure varchar(60), actual_departure varchar(60), scheduled_arrival varchar(60), actual_arrival varchar(60), status varchar(60), airline_code varchar(60))"""
cursor.execute(query)

In [84]:
columns = ",".join(flights_df.columns)
placeholders = ",".join(["%s"]*len(flights_df.columns))
query = f"insert into flight({columns}) values({placeholders})"
tuple_data = []
for i in flights_df.index:
    row = list(flights_df.loc[i].values)
    row = tuple(row)
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

NameError: name 'flights_df' is not defined

In [83]:
list_aircraft=flights_df['aircraft_registration'].unique().tolist()#294 entries
cleanlist_aircraft=[x for x in list_aircraft if x != "None"]
cleanlist_aircraft

NameError: name 'flights_df' is not defined

## 3. Steps to collect aircraft data using API and convert into SQL

In [ ]:
#Defining a function with icao code
def fetch_aircraft_by_reg(reg_code):
    url=f"https://aerodatabox.p.rapidapi.com/aircrafts/reg/{reg_code}"
    aircraft_response=requests.get(url,headers=headers)
    time.sleep(1)
    return aircraft_response.json()
    

In [ ]:
aircraft_all={}

for REG_code in cleanlist_aircraft[:100]:
    try:
        aircraft_data=fetch_aircraft_by_reg(REG_code)
        aircraft_all[REG_code]=aircraft_data
        print(REG_code, aircraft_all[REG_code])
    except Exception as e:
        print(f"Error loading airport {REG_code}: {e}")

G-XLED {'id': 1920, 'reg': 'G-XLED', 'active': True, 'serial': '144', 'hexIcao': '406A03', 'airlineName': 'British Airways', 'iataCodeShort': '388', 'icaoCode': 'A388', 'model': 'A388', 'modelCode': '380-841', 'numSeats': 469, 'rolloutDate': '2013-08-01', 'firstFlightDate': '2013-08-01', 'deliveryDate': '2014-01-17', 'registrationDate': '2014-01-17', 'typeName': 'Airbus A380-800', 'numEngines': 4, 'engineType': 'Jet', 'isFreighter': False, 'productionLine': 'Airbus A380', 'ageYears': 12.8, 'verified': True, 'numRegistrations': 1}
HL7203 {'id': 23455, 'reg': 'HL7203', 'active': True, 'serial': '60378', 'hexIcao': '71BA03', 'airlineName': 'Korean Air', 'iataCodeShort': '777', 'icaoCode': 'B777', 'model': 'B773', 'modelCode': 'B777-3B5ER', 'numSeats': 277, 'rolloutDate': '2018-02-06', 'firstFlightDate': '2018-02-07', 'deliveryDate': '2018-03-20', 'registrationDate': '2018-03-20', 'typeName': 'Boeing 777', 'numEngines': 2, 'engineType': 'Jet', 'isFreighter': False, 'productionLine': 'Boein

In [ ]:
import pandas as pd

def extract_aircraft_info(aircraft_data):
    extracted_aircraft_info = {
        'aircraft_code': aircraft_data.get('reg'),
        'aircraft_type': aircraft_data.get('typeName'),
        'serial': aircraft_data.get('serial'),
        'model': aircraft_data.get('model'),
        'airline': aircraft_data.get('airlineName'),
        'icao_code': aircraft_data.get('icaoCode'),
        'engine': aircraft_data.get('engineType'),
        'registration': aircraft_data.get('id')         
        }
    return extracted_aircraft_info

In [ ]:
extracted_aircrafts_data = []
for aircraft_code, data in aircraft_all.items():
        extracted_aircrafts_data.append(extract_aircraft_info(data))

# Create a pandas DataFrame from the extracted data
aircraft_df = pd.DataFrame(extracted_aircrafts_data)

aircraft_df_dropped = aircraft_df.dropna() #drops rows with null values since NaN and NaT are not accepteble in sql
display(aircraft_df_dropped)#dropped 2 rows

NameError: name 'aircraft_all' is not defined

In [ ]:
query = """create table if not exists Aircraft(aircraft_id int primary key AUTO_INCREMENT, aircraft_code varchar(60) UNIQUE, aircraft_type varchar(60), serial varchar(60), model varchar(60), airline varchar(60), icao_code varchar(60), engine varchar(60), registration int unique)"""
cursor.execute(query)
connection.commit()

In [ ]:
columns = ",".join(aircraft_df_dropped.columns)
placeholders = ",".join(["%s"]*len(aircraft_df_dropped.columns))
query = f"insert into Aircraft({columns}) values({placeholders})"
tuple_data = []
for i in aircraft_df_dropped.index:
    row = list(aircraft_df_dropped.loc[i].values)
    row[7]=int(row[7])
    row = tuple(row)
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

## 4. Steps to collect airport_delays data using API and convert into SQL

In [9]:
def fetch_airport_delays_data(icao_code):
    url=f"https://aerodatabox.p.rapidapi.com/airports/icao/{icao_code}/delays/2026-05-18T12:00/2026-05-19T00:00"
    airport_delays_response = requests.get(url, headers=headers)
    time.sleep(1)
    return airport_delays_response.json()
        

In [13]:
airport_delay_list = [
    "EGLL", "KJFK", "OMDB", "VIDP", "VOBL", "EGCC", "EDDM", "CYVR", "RJOO", "WMKP", "VTCC", "EHAM", "LFPG", "WSSS", "YSSY"

]

delay_data_all = {}

for icao in airport_delay_list:
    try:
        airport_delay_data = fetch_airport_delays_data(icao)
        delay_data_all[icao]= airport_delay_data
        print(airport_delay_data)
    except Exception as e:
        print(f"Error loading airport {icao}: {e}")

[{'airportIcao': 'EGLL', 'from': {'utc': '2026-05-18 09:00Z', 'local': '2026-05-18 10:00+01:00'}, 'to': {'utc': '2026-05-18 11:00Z', 'local': '2026-05-18 12:00+01:00'}, 'departuresDelayInformation': {'numTotal': 87, 'numQualifiedTotal': 87, 'numCancelled': 0, 'medianDelay': '00:26:00', 'delayIndex': 1.44}, 'arrivalsDelayInformation': {'numTotal': 76, 'numQualifiedTotal': 76, 'numCancelled': 0, 'medianDelay': '-00:12:00', 'delayIndex': 0.0}}, {'airportIcao': 'EGLL', 'from': {'utc': '2026-05-18 09:15Z', 'local': '2026-05-18 10:15+01:00'}, 'to': {'utc': '2026-05-18 11:15Z', 'local': '2026-05-18 12:15+01:00'}, 'departuresDelayInformation': {'numTotal': 85, 'numQualifiedTotal': 85, 'numCancelled': 0, 'medianDelay': '00:26:00', 'delayIndex': 1.44}, 'arrivalsDelayInformation': {'numTotal': 79, 'numQualifiedTotal': 79, 'numCancelled': 1, 'medianDelay': '-00:09:00', 'delayIndex': 0.0}}, {'airportIcao': 'EGLL', 'from': {'utc': '2026-05-18 09:30Z', 'local': '2026-05-18 10:30+01:00'}, 'to': {'utc'

In [21]:
import pandas as pd
def extract_delay_info(delay_data):
    extracted_delay_info_list=[]
    for dic in delay_data:
        extracted_delay_info = {
            'icao_code': dic.get('airportIcao'),
            'date': dic.get('from', {}).get('utc', '').split(" ")[0],
            'from_time': dic.get('from', {}).get('utc', '').split(" ")[1],
            'to_time': dic.get('to', {}).get('utc', '').split(" ")[1],
            'dep_median_delay': dic.get('departuresDelayInformation', {}).get('medianDelay', ''),
            'dep_cancelled_flight': dic.get('departuresDelayInformation', {}).get('numCancelled', ''),
            'arr_median_delay': dic.get('arrivalsDelayInformation', {}).get('medianDelay', ''),
            'arr_cancelled_flight': dic.get('arrivalsDelayInformation', {}).get('numCancelled', ''),
            'total_flights': dic.get('departuresDelayInformation', {}).get('numTotal', 0)
        
        }
        extracted_delay_info_list.append(extracted_delay_info)
    return extracted_delay_info_list
    
extracted_delay_data=[]
for icao, data in delay_data_all.items():
    extracted_delay_data.extend(extract_delay_info(data))
    #not using append because Since extract_delay_info(data) already returns a list, append() adds the whole list as one item

# Create DataFrame
delay_df = pd.DataFrame(extracted_delay_data)

display(delay_df)

,icao_code,date,from_time,to_time,dep_median_delay,dep_cancelled_flight,arr_median_delay,arr_cancelled_flight,total_flights
0,EGLL,2026-05-18,09:00Z,11:00Z,00:26:00,0,-00:12:00,0,87
1,EGLL,2026-05-18,09:15Z,11:15Z,00:26:00,0,-00:09:00,1,85
2,EGLL,2026-05-18,09:30Z,11:30Z,00:25:00,0,-00:08:00,1,86
3,EGLL,2026-05-18,09:45Z,11:45Z,00:25:00,0,-00:06:00,1,84
4,EGLL,2026-05-18,10:00Z,12:00Z,00:26:00,0,-00:07:00,1,77
...,...,...,...,...,...,...,...,...,...
730,YSSY,2026-05-18,11:00Z,13:00Z,00:00:00,1,00:01:00,0,11
731,YSSY,2026-05-18,11:15Z,13:15Z,,0,00:00:00,0,9
732,YSSY,2026-05-18,11:30Z,13:30Z,,0,-00:06:00,0,7
733,YSSY,2026-05-18,11:45Z,13:45Z,,0,00:03:00,0,7


In [29]:
delay_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 735 entries, 0 to 734
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   icao_code             735 non-null    str  
 1   date                  735 non-null    str  
 2   from                  735 non-null    str  
 3   to                    735 non-null    str  
 4   dep_median_delay      735 non-null    str  
 5   dep_cancelled_flight  735 non-null    int64
 6   arr_median_delay      735 non-null    str  
 7   arr_cancelled_flight  735 non-null    int64
 8   total_flights         735 non-null    int64
dtypes: int64(3), str(6)
memory usage: 51.8 KB


In [6]:
query="""create table if not exists airport_delay(delay_id int primary key auto_increment, icao_code varchar(60), date varchar(60), from_time varchar(60), to_time varchar(60), dep_median_delay varchar(60), dep_cancelled_flight int, arr_median_delay varchar(60), arr_cancelled_flight int, total_flights int)"""
cursor.execute(query)

In [22]:
columns = ','.join(delay_df.columns)
placeholders = ','.join(['%s']*len(delay_df.columns))
query = f"insert into airport_delay({columns}) values({placeholders})"
tuple_data = []
for i in delay_df.index:
    row = list(delay_df.loc[i].values)
    row[5]=int(row[5])
    row[7]=int(row[7])
    row[8]=int(row[8])
    row = tuple(row)
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

# SQL Queries

## 1. Show the total number of flights for each aircraft model, listing the model and its count.

In [ ]:
query = "SELECT ROW_NUMBER() OVER () AS no, model, COUNT(flight_number) AS Total_flights FROM aircraft JOIN flight ON aircraft_code=aircraft_registration  GROUP BY model"

df=pd.read_sql(query, connection)
print(df.to_string(index=False))
#verified manually using SELECT model, aircraft_code, aircraft_registration, COUNT( aircraft_registration) as total_flight FROM flight JOIN aircraft on aircraft_code=aircraft_registration where model = 'A20N' group by model, aircraft_code, aircraft_registration;

 no model  Total_flights
  1  B744              1
  2  A20N             17
  3  A320             16
  4  A21N             14
  5  AT75              2
  6  A321              6
  7  E170              1
  8  B738              8
  9  B789              7
 10  E175              1
 11  A319              5
 12  A388              7
 13  B773             12
 14  B772              1
 15  B781              1
 16   777              2
 17  A35K              1
 18  A333              2
 19  CRJ9              3
 20  B752              1
 21  A359              1
 22  B38M              5
 23  B788              2
 24  A332              1


C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\3108856341.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


## 2. List all aircraft (registration, model) that have been assigned to more than 5 flights

In [32]:
#since flight count is not more than 4, i used greater than or equal to 3
query = """SELECT aircraft_registration, model, COUNT(*) AS flight_count 
FROM aircraft JOIN flight 
ON aircraft_code = aircraft_registration 
GROUP BY aircraft_code, model 
HAVING COUNT(*) >=3"""
df=pd.read_sql(query, connection)
df


C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\3830913986.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,aircraft_registration,model,flight_count
0,G-EUYV,A320,4
1,G-XLED,A388,4
2,EI-SIA,A20N,3
3,4X-EDA,B789,3
4,G-EUYC,A320,3
5,D-AGWV,A319,4


## 3. For each airport, display its name and the number of outbound flights, but only for airports with more than 5 flights.

In [39]:
query = "SELECT icao_code, name, COUNT(flight_number) as number_of_outbound_flights from airport join flight on icao_code=origin_icao GROUP BY icao_code having count(flight_number)>5"
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\4238549280.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,icao_code,name,number_of_outbound_flights
0,YSSY,Sydney Kingsford Smith,20
1,VOBL,Bangalore Bengaluru,21
2,OMDB,Dubai,22
3,VIDP,New Delhi Indira Gandhi,21
4,EHAM,Amsterdam Schiphol,22
5,VTCC,Chiangmai Chiang Mai,20
6,EDDM,Munich,21
7,KJFK,New York John F Kennedy,21
8,EGLL,London Heathrow,22
9,CYVR,Vancouver,20


## 4. Find the top 3 destination airports (name, city) by number of arriving flights, sorted by count descending.

In [44]:
query = """SELECT destination_icao, name, city, COUNT(flight_number) AS arriving_flights
 FROM flight JOIN airport ON destination_icao = icao_code
   GROUP BY icao_code 
   ORDER BY arriving_flights DESC
     LIMIT 3"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\2126184087.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,destination_icao,name,city,arriving_flights
0,WSSS,Singapore Changi,Singapore,28
1,LFPG,Paris Charles de Gaulle,Paris,26
2,EGLL,London Heathrow,London,24


## 5. Show for each flight: number, origin, destination, and a label 'Domestic' or 'International' using CASE WHEN on country match

In [46]:
query = """SELECT flight_number, origin_icao, destination_icao,
CASE 
WHEN airport1.country = airport2.country 
THEN 'Domestic' ELSE 'International'
END AS flight_type
 FROM flight 
 JOIN airport AS airport1 on origin_icao=airport1.icao_code 
  JOIN airport AS airport2 on destination_icao = airport2.icao_code """
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\1675702155.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,flight_number,origin_icao,destination_icao,flight_type
0,6E 1657,VIDP,OMDB,International
1,6E 3223,EHAM,LFPG,International
2,6E 817,VOBL,VIDP,Domestic
3,6E 861,VIDP,VOBL,Domestic
4,AA 141,EGLL,KJFK,International
5,AA 292,KJFK,VIDP,International
6,AC 5944,WSSS,WMKP,International
7,AC 5948,WMKP,WSSS,International
8,AC 7610,OMDB,WSSS,International
9,AC 860,CYVR,EGLL,International


## 6. Show the 5 most recent arrivals at “DEL” airport including flight number, aircraft, departure airport name, and arrival time, ordered by latest arrival.

In [ ]:
query = """SELECT city, flight_number, aircraft_code AS aircraft, name AS departure_airport_name, 
actual_arrival, DATE_FORMAT(STR_TO_DATE(actual_arrival, '%Y-%m-%d %H:%iZ'), '%H:%i') AS arrival_time 
FROM airport 
JOIN flight ON airport.icao_code = flight.origin_icao
JOIN aircraft ON aircraft.aircraft_code=flight.aircraft_registration 
WHERE iata_code = 'DEL'
ORDER BY arrival_time DESC
LIMIT 5"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\1803676126.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,city,flight_number,aircraft,departure_airport_name,actual_arrival,arrival_time
0,New Delhi,AI 358,VT-TSN,New Delhi Indira Gandhi,2026-05-17 22:08Z,22:08
1,New Delhi,MU 564,B-8226,New Delhi Indira Gandhi,2026-05-17 20:11Z,20:11
2,New Delhi,6E 1701,VT-IPG,New Delhi Indira Gandhi,2026-05-17 20:00Z,20:00
3,New Delhi,6E 1657,VT-IPY,New Delhi Indira Gandhi,2026-05-17 18:12Z,18:12
4,New Delhi,6E 772,VT-IWN,New Delhi Indira Gandhi,2026-05-17 18:00Z,18:00


## 7. Find all airports with no arriving flights (never used as a destination in flights table)

In [63]:
query = """SELECT icao_code, name, city
FROM airport
LEFT JOIN flight ON airport.icao_code = flight.destination_icao
WHERE flight.destination_icao= 'None'"""
df= pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\2695690141.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df= pd.read_sql(query, connection)


,icao_code,name,city


## 8. For each airline, count the number of flights by status (e.g., 'On Time', 'Delayed', 'Cancelled') using CASE WHEN.

In [82]:
query = """WITH base AS (SELECT aircraft.airline,
        STR_TO_DATE(flight.scheduled_departure, '%Y-%m-%d %H:%iZ') AS sch_dep,
        STR_TO_DATE(flight.actual_departure, '%Y-%m-%d %H:%iZ') AS act_dep,
        STR_TO_DATE(flight.scheduled_arrival, '%Y-%m-%d %H:%iZ') AS sch_arr,
        STR_TO_DATE(flight.actual_arrival, '%Y-%m-%d %H:%iZ') AS act_arr 
        FROM flight
        JOIN aircraft ON aircraft.aircraft_code = flight.aircraft_registration)        
    SELECT airline,
    SUM(CASE 
        WHEN TIMESTAMPDIFF(MINUTE, sch_dep, act_dep) > 0 
        THEN 1 ELSE 0
    END) AS delayed_departure,
    SUM(CASE 
        WHEN TIMESTAMPDIFF(MINUTE, sch_dep, act_dep) <= 0 
        THEN 1 ELSE 0
    END) AS on_time_departure,
    SUM(CASE 
        WHEN TIMESTAMPDIFF(MINUTE, sch_arr, act_arr) > 0 
        THEN 1 ELSE 0
    END) AS delayed_arrival,
    SUM(CASE 
        WHEN TIMESTAMPDIFF(MINUTE, sch_arr, act_arr) <= 0 
        THEN 1 ELSE 0
    END) AS on_time_arrival
FROM base
Group BY airline"""

df= pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\2336441354.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df= pd.read_sql(query, connection)


,airline,delayed_departure,on_time_departure,delayed_arrival,on_time_arrival
0,UPS,0.0,1.0,0.0,0.0
1,IndiGo,2.0,15.0,3.0,13.0
2,Alliance Air,0.0,1.0,0.0,1.0
3,American Airlines,5.0,0.0,0.0,6.0
4,Republic Airways,2.0,0.0,0.0,2.0
5,British Airways,15.0,1.0,9.0,7.0
6,Air India,2.0,3.0,1.0,4.0
7,Vistara,0.0,1.0,0.0,1.0
8,SAS Ireland,3.0,0.0,0.0,3.0
9,El Al,0.0,3.0,0.0,3.0


## 9. Show all cancelled flights, with aircraft and both airports, ordered by departure time descending

## 10. List all city pairs (origin-destination) that have more than 2 different aircraft models operating flights between them

In [7]:
query = """SELECT airport_departure.city AS origin_city,
    airport_arrival.city AS destination_city,
    COUNT(DISTINCT aircraft.model) AS aircraft_model_count
FROM flight JOIN airport airport_departure
    ON flight.origin_icao = airport_departure.icao_code
JOIN airport airport_arrival ON flight.destination_icao = airport_arrival.icao_code
JOIN aircraft ON flight.aircraft_registration = aircraft.aircraft_code
GROUP BY airport_departure.city, airport_arrival.city 
HAVING COUNT(DISTINCT aircraft.model) >= 2"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_20248\2828516602.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,origin_city,destination_city,aircraft_model_count
0,Bangalore,New Delhi,2
1,London,New York,2
2,New Delhi,Bangalore,2


## 11. For each destination airport, compute the % of delayed flights (status='Delayed') among all arrivals, sorted by highest percentage

In [8]:
query = """WITH base AS (SELECT
        flight.destination_icao,
        STR_TO_DATE(flight.scheduled_arrival, '%Y-%m-%d %H:%iZ') AS sch_arr,
        STR_TO_DATE(flight.actual_arrival, '%Y-%m-%d %H:%iZ') AS act_arr
    FROM flight)
SELECT destination_icao,
    COUNT(*) AS total_flights,
    SUM(CASE 
        WHEN TIMESTAMPDIFF(MINUTE, sch_arr, act_arr) > 0 
        THEN 1 ELSE 0 
        END) AS delayed_flights,
    ROUND(100.0 * SUM(CASE 
            WHEN TIMESTAMPDIFF(MINUTE, sch_arr, act_arr) > 0 
            THEN 1 ELSE 0 
            END) / COUNT(*), 2) AS delayed_percentage
FROM base GROUP BY destination_icao ORDER BY delayed_percentage DESC"""
df=pd.read_sql(query, connection)
df


C:\Users\hamsa\AppData\Local\Temp\ipykernel_20248\1103599209.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,destination_icao,total_flights,delayed_flights,delayed_percentage
0,KLAS,1,1.0,100.0
1,CYQQ,1,1.0,100.0
2,KSFO,2,2.0,100.0
3,LROP,5,5.0,100.0
4,LOWW,2,2.0,100.0
...,...,...,...,...
134,LTFM,1,0.0,0.0
135,YPPH,1,0.0,0.0
136,LRTR,1,0.0,0.0
137,LEBL,1,0.0,0.0
